# Semana 14: reproducibilidad mínima

Pipeline, métricas, artefactos y manifest reproducible.

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
RANDOM_STATE = 42
output_dir = Path("artifacts_semana14")
output_dir.mkdir(exist_ok=True)

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)
proba = pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
metrics = {"f1": float(f1_score(y_test, pred)), "roc_auc": float(roc_auc_score(y_test, proba))}
metrics

In [ ]:
model_path = output_dir / "model.joblib"
metrics_path = output_dir / "metrics.json"
dump(pipeline, model_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(model_path, metrics_path)

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "params": {"random_state": RANDOM_STATE, "model": "LogisticRegression", "threshold": 0.5},
    "metrics": metrics,
    "artifacts": {path.name: file_sha256(path) for path in [model_path, metrics_path]},
}
manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

## Cierre

El manifest permite identificar qué parámetros, métricas y archivos sustentan la decisión.